# Camping Person Account data quality assessment

This notebook evaluates the raw source worksheet (`Document with the data`) of the Camping migration workbook.

## Stages

1. **Quality overview** — profile the source data and measure Person Account contract readiness.
2. **Rescuing** — normalize and repair recoverable data-quality problems.
3. **Comparison** — compare cleaned records with Salesforce data downloaded into MySQL.
4. **Import** — identify the final importable population and prepare it for import.

## Scope

- Use worksheet 2 as the source of truth; worksheet 1 contains broken Excel transformations.
- Preserve the source workbook without modification.
- Validate the Salesforce Person Account contract.
- Exclude Loyalty Member and loyalty-points processing.
- Retain `consent_central` for later analysis.
- Keep summaries inside this notebook.

## Terminology

- **Contract-ready:** passes the locally encoded Person Account requirements.
- **Final importable:** contract-ready and appropriately classified after comparison with Salesforce.

In [ ]:
from datetime import date

import pandera.polars as pa
import phonenumbers
import polars as pl
import polars.selectors as cs
import pycountry
from nameparser import HumanName
from pandera.polars import PolarsData

WORKBOOK_PATH = "data/source/2607_Data cleaned - Grubhof.xlsx"

In [ ]:
COLUMN_RENAMES = {
    "NA_KURZ": "external_id",
    "NA_ANREDE": "salutation",
    "NA_NAME2": "first_name",
    "NA_NAME3": "middle_name",
    "NA_NAME1": "last_name",
    "NA_GEBDAT": "birth_date",
    "NA_TELEX": "email",
    "NA_TEL1": "phone",
    "NZ_SPRACHE": "preferred_language",
    "NA_STR": "address",
    "NA_CPLZ": "postal_code",
    "NA_ORT": "city",
    "NA_INT": "country",
    "NA_MAILING": "consent_camping",
}

source_data = pl.read_excel(
    WORKBOOK_PATH,
    sheet_id=2,
)
raw = (
    source_data
    .rename(COLUMN_RENAMES)
    .select(COLUMN_RENAMES.values())
)
profiled = raw.with_columns(cs.string().replace("", None))

In [ ]:
blank_row = pl.all_horizontal(pl.all().is_null())
contract_data = profiled.filter(~blank_row)
data = contract_data

pl.DataFrame(
    {
        "worksheet_rows": [profiled.height],
        "blank_rows": [profiled.height - data.height],
        "data_rows": [data.height],
    }
)

In [ ]:
raw.glimpse(max_items_per_column=5)

In [ ]:
analysis = (
    data.null_count()
    .transpose(include_header=True, column_names=["null_count"])
    .join(
        data.select(pl.all().drop_nulls().n_unique()).transpose(
            include_header=True, column_names=["cardinality"]
        ),
        on="column",
    )
    .with_columns(
        (data.height - pl.col("null_count")).alias("present_rows")
    )
    .with_columns(
        (100 * pl.col("present_rows") / data.height).alias("completeness_pct")
    )
    .select(
        "column",
        "present_rows",
        "null_count",
        "completeness_pct",
        "cardinality",
    )
    .sort("completeness_pct", "column")
)

with pl.Config(tbl_rows=-1):
    print(analysis)

Some observations:

1. Worksheet 2 provides 146,050 first names directly.
2. Middle name has 390 populated source values and can be explored separately.
3. Only directly renamed source fields are included; no target defaults are manufactured here.

In [ ]:
blank_columns = (
    analysis
    .filter(pl.col("present_rows") == 0)
    .get_column("column")
    .to_list()
)

print(f"Removing {len(blank_columns)} blank columns:")
for name in blank_columns:
    print(f"  - {name}")

data = data.drop(blank_columns)

pl.DataFrame(
    {
        "blank_columns_removed": [len(blank_columns)],
        "remaining_columns": [data.width],
    }
)

In [ ]:
data.glimpse()

In [ ]:
(
    data
    .select("middle_name")
    .group_by("middle_name")
    .len(name="rows")
    .sort("rows", "middle_name", descending=[True, False])
)

In [ ]:
print("Removing middle_name")
data = data.drop("middle_name")
data.width

In [ ]:
print(
    "Exact duplicate data rows beyond the first occurrence: "
    f"{data.height - data.n_unique():,}"
)

In [ ]:
categorical_distribution = (
    profiled.select(
        cs.by_name(
            "salutation",
            "gender",
            "preferred_language",
            require_all=False,
        )
        | cs.ends_with("_customer")
        | cs.starts_with("consent_")
    )
    .unpivot(variable_name="column")
    .filter(pl.col("value").is_not_null())
    .group_by("column", "value")
    .len(name="rows")
    .with_columns(
        (100 * pl.col("rows") / pl.col("rows").sum().over("column"))
        .alias("pct_of_present")
    )
    .sort(["column", "rows", "value"], descending=[False, True, False])
)

categorical_distribution

In [ ]:
import matplotlib.pyplot as plt

MIN_BIRTH_YEAR = 1900
MAX_BIRTH_YEAR = 2020

birth_years = (
    data
    .select(
        pl.col("birth_date")
        .dt.year()
        .alias("birth_year")
    )
    .drop_nulls()
)

out_of_range = birth_years.filter(
    (pl.col("birth_year") < MIN_BIRTH_YEAR)
    | (pl.col("birth_year") > MAX_BIRTH_YEAR)
)
print(
    f"Excluding {out_of_range.height:,} birth years outside "
    f"{MIN_BIRTH_YEAR}-{MAX_BIRTH_YEAR}: "
    f"{sorted(out_of_range.get_column('birth_year').unique().to_list())}"
)

birth_year_counts = (
    birth_years
    .filter(
        pl.col("birth_year").is_between(MIN_BIRTH_YEAR, MAX_BIRTH_YEAR)
    )
    .group_by("birth_year")
    .len(name="rows")
    .sort("birth_year")
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(
    birth_year_counts["birth_year"].to_list(),
    birth_year_counts["rows"].to_list(),
    width=1.0,
    color="#4C78A8",
)
ax.set_title("Birth year distribution")
ax.set_xlabel("Birth year")
ax.set_ylabel("Rows")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---

## Data Transforms

In [ ]:
transformed = data.with_columns(
    cs.string().str.strip_chars().replace("", None)
)

In [ ]:
LEGAL_ENTITY_PATTERN = (
    r"(?i)(?:"
    r"\bgmbh\b|\bco\.?\s*kg\b|\bltd\.?|\blimited\b|\bllc\b|"
    r"\binc\.?|\bag\b|\bkg\b|\bog\b|"
    r"\bb\.?\s*v\.?(?:\s|$)|\be\.?\s*v\.?(?:\s|$)"
    r")"
)
ORGANISATION_PATTERN = (
    r"(?i)\b(?:"
    r"verein|verband|club|stiftung|gemeinde|reisebüro|"
    r"travel\s+group|camping|caravan|hotel|verlag|touristik|union"
    r")\b"
)
HOUSEHOLD_PATTERN = (
    r"(?i)(?:"
    r"\bfamilie\b|\bfamily\b|\bfam\.?(?:\s|$)|\beheleute\b"
    r")"
)
NAME_PLACEHOLDERS = [
    "test",
    "unknown",
    "n/a",
    "none",
    "na",
    "null",
    "xxx",
    "first",
    "firstname",
    "first name",
    "last",
    "lastname",
    "last name",
    "-",
    ".",
]
EXCLUSION_RULES = [
    "legal_entity",
    "organisation",
    "household",
    "placeholder",
    "contains_digit",
]

full_name = pl.concat_str(
    "first_name",
    "last_name",
    separator=" ",
    ignore_nulls=True,
).str.strip_chars()

name_exclusion_flags = (
    transformed
    .select(
        "external_id",
        "first_name",
        "last_name",
        full_name.alias("full_name"),
    )
    .with_columns(
        pl.col("full_name")
        .str.contains(LEGAL_ENTITY_PATTERN)
        .alias("legal_entity"),
        pl.col("full_name")
        .str.contains(ORGANISATION_PATTERN)
        .alias("organisation"),
        pl.col("full_name")
        .str.contains(HOUSEHOLD_PATTERN)
        .alias("household"),
        pl.any_horizontal(
            pl.col("first_name").str.to_lowercase().is_in(NAME_PLACEHOLDERS),
            pl.col("last_name").str.to_lowercase().is_in(NAME_PLACEHOLDERS),
        ).alias("placeholder"),
        pl.col("full_name").str.contains(r"\d").alias("contains_digit"),
    )
    .with_columns(
        pl.any_horizontal(cs.boolean())
        .fill_null(False)
        .alias("exclude")
    )
)

transformed = transformed.with_columns(
    name_exclusion_flags.get_column("exclude")
)

exclusion_summary = (
    name_exclusion_flags
    .select(EXCLUSION_RULES)
    .unpivot(variable_name="rule", value_name="matched")
    .filter("matched")
    .group_by("rule")
    .len(name="rows")
    .sort("rows", descending=True)
)

print("Potential non-person records by rule")
print(exclusion_summary)
print(
    "\nTotal rows flagged for exclusion: "
    f"{transformed.get_column('exclude').sum():,}"
)
print("\nExample flagged rows")
name_exclusion_flags.filter("exclude").head(20)

In [ ]:
def capitalize_first_name(value: str) -> str:
    name = HumanName(first=value)
    name.capitalize()
    return name.first


def capitalize_last_name(value: str) -> str:
    name = HumanName(last=value)
    name.capitalize()
    return name.last


names_before = transformed.select(
    "first_name",
    "last_name",
)

transformed = transformed.with_columns(
    pl.when(~pl.col("exclude"))
    .then(
        pl.col("first_name").map_elements(
            capitalize_first_name,
            return_dtype=pl.String,
        )
    )
    .otherwise(pl.col("first_name"))
    .alias("first_name"),
    pl.when(~pl.col("exclude"))
    .then(
        pl.col("last_name").map_elements(
            capitalize_last_name,
            return_dtype=pl.String,
        )
    )
    .otherwise(pl.col("last_name"))
    .alias("last_name"),
)

name_changes = pl.DataFrame(
    {
        "first_name_before": names_before["first_name"],
        "first_name_after": transformed["first_name"],
        "last_name_before": names_before["last_name"],
        "last_name_after": transformed["last_name"],
        "exclude": transformed["exclude"],
    }
).filter(
    pl.any_horizontal(
        pl.col("first_name_before") != pl.col("first_name_after"),
        pl.col("last_name_before") != pl.col("last_name_after"),
    )
)

print(
    "First names capitalized: "
    f"{name_changes.filter(pl.col('first_name_before') != pl.col('first_name_after')).height:,}"
)
print(
    "Last names capitalized: "
    f"{name_changes.filter(pl.col('last_name_before') != pl.col('last_name_after')).height:,}"
)
print("\nExample name changes")
name_changes.head(20)

In [ ]:
PLACEHOLDER_BIRTH_DATES = [
    date(1899, 12, 31),
    date(1900, 1, 1),
]

placeholder_birth_date_summary = (
    pl.DataFrame({"birth_date": PLACEHOLDER_BIRTH_DATES})
    .join(
        transformed
        .filter(
            pl.col("birth_date").is_in(PLACEHOLDER_BIRTH_DATES)
        )
        .group_by("birth_date")
        .len(name="rows"),
        on="birth_date",
        how="left",
    )
    .with_columns(
        pl.col("rows").fill_null(0)
    )
)

transformed = transformed.with_columns(
    pl.col("birth_date").replace(
        PLACEHOLDER_BIRTH_DATES,
        None,
    )
)

print("Placeholder birth dates converted to null")
placeholder_birth_date_summary

In [ ]:
ISO_639_1_CODES = sorted(
    language.alpha_2
    for language in pycountry.languages
    if hasattr(language, "alpha_2")
)

languages_before = transformed.get_column("preferred_language")

transformed = transformed.with_columns(
    pl.col("preferred_language")
    .str.to_lowercase()
    .replace({"cz": "cs"})
)

language_change_summary = (
    pl.DataFrame(
        {
            "before": languages_before,
            "after": transformed["preferred_language"],
        }
    )
    .filter(
        pl.col("before") != pl.col("after")
    )
    .group_by("before", "after")
    .len(name="rows")
    .sort("rows", descending=True)
)

invalid_language_codes = (
    transformed
    .filter(
        pl.col("preferred_language").is_not_null()
        & ~pl.col("preferred_language").is_in(ISO_639_1_CODES)
    )
    .group_by("preferred_language")
    .len(name="rows")
    .sort("rows", descending=True)
)

print("Preferred-language corrections")
print(language_change_summary)
print("\nRemaining non-ISO 639-1 language codes")
invalid_language_codes

In [ ]:
CONSENT_CAMPING_VALUES = {
    "J": True,
    "N": False,
}

consent_camping_before = (
    transformed
    .group_by("consent_camping")
    .len(name="rows")
    .sort("rows", descending=True)
)

transformed = transformed.with_columns(
    pl.col("consent_camping").replace_strict(
        CONSENT_CAMPING_VALUES
    )
)

consent_camping_after = (
    transformed
    .group_by("consent_camping")
    .len(name="rows")
    .sort("rows", descending=True)
)

print("Consent Camping before conversion")
print(consent_camping_before)
print("\nConsent Camping after conversion")
consent_camping_after

In [ ]:
def normalize_phone(
    row: dict[str, str | None],
) -> str | None:
    value = row["phone"]

    if value is None:
        return None

    if not phonenumbers.is_possible_number_string(
        value,
        row["country"],
    ):
        return value

    return phonenumbers.format_number(
        phonenumbers.parse(
            value,
            row["country"],
        ),
        phonenumbers.PhoneNumberFormat.E164,
    )


phones_before = transformed.get_column("phone")

transformed = transformed.with_columns(
    pl.struct("phone", "country")
    .map_elements(
        normalize_phone,
        return_dtype=pl.String,
    )
    .alias("phone")
)

phone_changes = (
    pl.DataFrame(
        {
            "before": phones_before,
            "after": transformed["phone"],
            "country": transformed["country"],
        }
    )
    .filter(
        pl.col("before") != pl.col("after")
    )
)

print(
    "Primary phones normalized to E.164: "
    f"{phone_changes.height:,}"
)
print("\nExample phone changes")
phone_changes.head(20)

---

## Person Account contract checks

The following checks measure source validity and current payload readiness.

In [ ]:
VALIDATION_DATE = date.today()
SALUTATIONS = ["Mr.", "Mrs.", "Ms.", "Dr.", "Prof."]
MATCHING_FIELDS = ["first_name", "last_name", "birth_date", "email"]
EMAIL_PATTERN = (
    r"^[0-9a-zA-Z]([-.\w]*[0-9a-zA-Z_+])*@"
    r"([0-9a-zA-Z][-\w]*[0-9a-zA-Z]\.)+"
    r"[a-zA-Z]{2,9}$"
)


def text_column(
    max_length: int,
    *,
    nullable: bool = True,
    required: bool = False,
) -> pa.Column:
    return pa.Column(
        pl.String,
        checks=pa.Check.str_length(max_value=max_length),
        nullable=nullable,
        required=required,
    )


def possible_phone(data: PolarsData) -> pl.LazyFrame:
    return data.lazyframe.select(
        pl.struct(data.key, "country").map_elements(
            lambda row: (
                row[data.key] is None
                or phonenumbers.is_possible_number_string(
                    row[data.key],
                    row["country"],
                )
            ),
            return_dtype=pl.Boolean,
        )
    )


person_account_source_schema = pa.DataFrameSchema(
    {
        "external_id": text_column(40, required=True),
        "cluster_id": text_column(40),
        "entra_external_id": text_column(255),
        "salutation": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=40),
                pa.Check.isin(
                    SALUTATIONS,
                    name="valid_salutation",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "first_name": text_column(40),
        "middle_name": text_column(40),
        "last_name": text_column(
            80,
            nullable=False,
            required=True,
        ),
        "birth_date": pa.Column(
            pl.Date,
            checks=[
                pa.Check.not_equal_to(
                    date(1899, 12, 31),
                    name="not_1899_placeholder",
                ),
                pa.Check.not_equal_to(
                    date(1900, 1, 1),
                    name="not_1900_placeholder",
                ),
                pa.Check.less_than_or_equal_to(
                    VALIDATION_DATE,
                    name="not_in_future",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "birth_place": text_column(255),
        "gender": text_column(50),
        "email": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=255),
                pa.Check.str_matches(
                    EMAIL_PATTERN,
                    name="valid_email",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "phone": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=50),
                pa.Check(
                    possible_phone,
                    name="possible_phone",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "preferred_language": pa.Column(
            pl.String,
            checks=[
                pa.Check.str_length(max_value=10),
                pa.Check.isin(
                    ISO_639_1_CODES,
                    name="valid_iso_639_1",
                ),
            ],
            nullable=True,
            required=False,
        ),
        "nationality_country_code": text_column(10),
        "address": text_column(255),
        "postal_code": text_column(20),
        "city": text_column(40),
        "state": text_column(80),
        "country": text_column(80),
        "invest_customer": pa.Column(
            pl.Boolean,
            nullable=True,
            required=False,
        ),
        "investment_status": text_column(50),
        "investment_expiration_date": pa.Column(
            pl.Date,
            nullable=True,
            required=False,
        ),
    },
    name="person_account_source",
)

person_account_transformed_schema = (
    person_account_source_schema.add_columns(
        {
            "consent_camping": pa.Column(
                pl.Boolean,
                nullable=True,
                required=True,
            ),
            "exclude": pa.Column(
                pl.Boolean,
                nullable=False,
                required=True,
            ),
        }
    )
)

person_account_matching_columns = {
    field: person_account_transformed_schema.columns[field]
    for field in MATCHING_FIELDS
}
person_account_matching_schema = pa.DataFrameSchema(
    person_account_matching_columns,
    name="person_account_matching",
).update_columns(
    {field: {"nullable": False} for field in MATCHING_FIELDS}
)

person_account_payload_columns = person_account_source_schema.update_columns(
    {"external_id": {"nullable": False}}
).columns
person_account_payload_schema = pa.DataFrameSchema(
    {
        "source": text_column(
            50,
            nullable=False,
            required=True,
        ),
        **person_account_payload_columns,
    },
    name="person_account_payload",
    unique=["source", "external_id"],
)

In [ ]:
FAILURE_CASES_SCHEMA = {
    "failure_case": pl.String,
    "schema_context": pl.String,
    "column": pl.String,
    "check": pl.String,
    "check_number": pl.Int64,
    "index": pl.Int64,
}


def collect_failure_cases(
    schema: pa.DataFrameSchema,
    frame: pl.DataFrame,
) -> pl.DataFrame:
    try:
        schema.validate(frame, lazy=True)
    except pa.errors.SchemaErrors as error:
        return error.failure_cases

    return pl.DataFrame(schema=FAILURE_CASES_SCHEMA)

In [ ]:
def summarize_failures(
    failures: pl.DataFrame,
    dataset: str,
) -> pl.DataFrame:
    return (
        failures
        .group_by("column", "check")
        .len(name="failures")
        .with_columns(
            pl.lit(dataset).alias("dataset")
        )
        .select(
            "dataset",
            "column",
            "check",
            "failures",
        )
    )


raw_source_failures = collect_failure_cases(
    person_account_source_schema,
    contract_data,
)
transformed_source_failures = collect_failure_cases(
    person_account_transformed_schema,
    transformed,
)
payload_failures = collect_failure_cases(
    person_account_payload_schema,
    transformed,
)

source_failure_summary = (
    pl.concat(
        [
            summarize_failures(
                raw_source_failures,
                "raw",
            ),
            summarize_failures(
                transformed_source_failures,
                "transformed",
            ),
        ]
    )
    .sort(
        "dataset",
        "failures",
        descending=[False, True],
    )
)

payload_failure_summary = (
    payload_failures
    .with_columns(
        pl.when(pl.col("check") == "column_in_dataframe")
        .then(pl.col("failure_case"))
        .otherwise(pl.col("column"))
        .alias("column")
    )
    .group_by("column", "check")
    .len(name="failures")
    .sort("failures", descending=True)
)

raw_source_invalid_indices = (
    raw_source_failures
    .get_column("index")
    .drop_nulls()
    .unique()
)
transformed_source_invalid_indices = (
    transformed_source_failures
    .get_column("index")
    .drop_nulls()
    .unique()
)
payload_invalid_indices = (
    payload_failures
    .get_column("index")
    .drop_nulls()
    .unique()
)
payload_missing_columns = (
    payload_failures
    .filter(
        pl.col("check") == "column_in_dataframe"
    )
    .get_column("failure_case")
    .unique()
    .to_list()
)

raw_duplicate_rows = (
    contract_data
    .with_row_index("index")
    .filter(
        ~pl.struct(contract_data.columns).is_first_distinct()
    )
    .get_column("index")
)
transformed_duplicate_rows = (
    transformed
    .with_row_index("index")
    .filter(
        ~pl.struct(transformed.columns).is_first_distinct()
    )
    .get_column("index")
)

raw_rescue_indices = (
    pl.concat(
        [
            raw_source_invalid_indices.to_frame(),
            raw_duplicate_rows.to_frame(),
        ],
        how="vertical_relaxed",
    )
    .get_column("index")
    .unique()
)
rescue_indices = (
    pl.concat(
        [
            transformed_source_invalid_indices.to_frame(),
            transformed_duplicate_rows.to_frame(),
        ],
        how="vertical_relaxed",
    )
    .get_column("index")
    .unique()
)

validation_summary = pl.DataFrame(
    {
        "measure": [
            "data_rows",
            "raw_source_rule_invalid_rows",
            "transformed_source_rule_invalid_rows",
            "raw_duplicate_rows_after_first",
            "transformed_duplicate_rows_after_first",
            "raw_combined_rows_requiring_rescue",
            "transformed_combined_rows_requiring_rescue",
            "raw_source_valid_unique_rows",
            "transformed_source_valid_unique_rows",
            "rows_flagged_exclude",
            "payload_ready_rows",
        ],
        "rows": [
            contract_data.height,
            raw_source_invalid_indices.len(),
            transformed_source_invalid_indices.len(),
            raw_duplicate_rows.len(),
            transformed_duplicate_rows.len(),
            raw_rescue_indices.len(),
            rescue_indices.len(),
            contract_data.height - raw_rescue_indices.len(),
            transformed.height - rescue_indices.len(),
            transformed.get_column("exclude").sum(),
            (
                0
                if payload_missing_columns
                else transformed.height
                - payload_invalid_indices.len()
            ),
        ],
    }
)

print("Source contract failures before and after transforms")
print(source_failure_summary)
print("\nPayload contract failures after transforms")
print(payload_failure_summary)
print(f"Missing payload columns: {payload_missing_columns}")
print("\nPerson Account validation summary")
validation_summary

In [ ]:
matching_failures = collect_failure_cases(
    person_account_matching_schema,
    transformed,
)

matching_presence = (
    transformed
    .select(
        pl.col(MATCHING_FIELDS).is_not_null().sum()
    )
    .unpivot(
        variable_name="field",
        value_name="present",
    )
)

matching_invalid = (
    matching_failures
    .group_by("column")
    .agg(
        pl.col("index").n_unique().alias("invalid")
    )
    .rename({"column": "field"})
)

matching_field_summary = (
    matching_presence
    .join(
        matching_invalid,
        on="field",
    )
    .with_columns(
        (
            transformed.height - pl.col("present")
        ).alias("missing"),
        (
            transformed.height - pl.col("invalid")
        ).alias("valid"),
    )
    .select(
        "field",
        "present",
        "missing",
        "valid",
        "invalid",
    )
)

matching_presence_counts = transformed.select(
    pl.len().alias("data_rows"),
    pl.col("first_name")
    .is_not_null()
    .sum()
    .alias("first_name_present"),
    pl.all_horizontal(
        pl.col("first_name").is_not_null(),
        pl.col("last_name").is_not_null(),
    )
    .sum()
    .alias("first_and_last_present"),
    pl.all_horizontal(
        pl.col("first_name").is_not_null(),
        pl.col("last_name").is_not_null(),
        pl.col("birth_date").is_not_null(),
    )
    .sum()
    .alias("names_and_birth_date_present"),
    pl.all_horizontal(
        pl.col("first_name").is_not_null(),
        pl.col("last_name").is_not_null(),
        pl.col("birth_date").is_not_null(),
        pl.col("email").is_not_null(),
    )
    .sum()
    .alias("all_four_present"),
).row(0)

matching_ready_rows = (
    transformed.height
    - matching_failures
    .get_column("index")
    .n_unique()
)

matching_population_funnel = pl.DataFrame(
    {
        "measure": [
            "data_rows",
            "first_name_present",
            "first_and_last_present",
            "names_and_birth_date_present",
            "all_four_present",
            "all_four_valid",
        ],
        "rows": [
            *matching_presence_counts,
            matching_ready_rows,
        ],
    }
)

print("Transformed matching fields: presence and validity")
print(matching_field_summary)
print("\nTransformed Salesforce matching population funnel")
matching_population_funnel

In [ ]:
transformation_checks = pl.DataFrame(
    {
        "check": [
            "row_count_preserved",
            "external_id_order_preserved",
            "flagged_names_unchanged",
            "placeholder_birth_dates_removed",
            "languages_are_iso_639_1",
            "consent_camping_is_boolean",
            "exclude_is_boolean",
            "exclude_is_non_null",
        ],
        "passed": [
            transformed.height == data.height,
            transformed["external_id"].equals(
                data["external_id"]
            ),
            name_changes.filter("exclude").is_empty(),
            transformed
            .filter(
                pl.col("birth_date").is_in(
                    PLACEHOLDER_BIRTH_DATES
                )
            )
            .is_empty(),
            invalid_language_codes.is_empty(),
            (
                transformed.schema["consent_camping"]
                == pl.Boolean
            ),
            transformed.schema["exclude"] == pl.Boolean,
            transformed["exclude"].null_count() == 0,
        ],
    }
)

print(
    "Transformation checks passed: "
    f"{transformation_checks.get_column('passed').sum()}"
    f"/{transformation_checks.height}"
)
transformation_checks

### First name, email and consent scenario

In [ ]:
first_name_valid = (
    pl.col("first_name").is_not_null()
    & (pl.col("first_name").str.len_chars() <= 40)
)

email_valid = (
    pl.col("email").is_not_null()
    & (pl.col("email").str.len_chars() <= 255)
    & pl.col("email").str.contains(EMAIL_PATTERN)
)

consent_granted = pl.col("consent_camping").eq(True)

first_email_consent_summary = (
    transformed.select(
        pl.len().alias("Data rows"),
        first_name_valid.sum().alias("Valid first name"),
        email_valid.sum().alias("Valid email"),
        consent_granted.sum().alias("Consent granted"),
        pl.all_horizontal(
            first_name_valid,
            email_valid,
            consent_granted,
        )
        .fill_null(False)
        .sum()
        .alias("Valid first name, email and consent"),
    )
    .unpivot(
        variable_name="Measure",
        value_name="Rows",
    )
)

first_email_consent_summary